# Baseline Benchmark and Evaluation Metrics

This notebook reproduces the standardized baseline benchmark for SDSS17 stellar classification. It calls reusable code from `src.baseline_benchmark` so the notebook and command-line benchmark share one implementation.

## Public Sources

- Logistic Regression and Random Forest project: https://github.com/wonhee3472/Stellar_Classification
- Random Forest notebook: https://www.kaggle.com/code/leylamammadzada/stellar-classification-random-forest
- CatBoost notebook: https://www.kaggle.com/code/aerdem/stellar-classification-catboost-97-accuracy
- Original dataset: https://www.kaggle.com/datasets/fedesoriano/stellar-classification-dataset-sdss17

Published scores from these sources are not directly comparable with this benchmark because they may use different splits, features and preprocessing. Here the same prepared train and validation splits are used for all models.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

repo_root = Path.cwd()
if not (repo_root / "data").exists() and (repo_root.parent / "data").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.baseline_benchmark import (
    CLASS_ORDER,
    SUMMARY_COLUMNS,
    load_data,
    prepare_features,
    resolve_paths,
    run_benchmark,
    validate_data,
)

paths = resolve_paths(repo_root)

## Load and Validate Prepared Data

The reusable loader validates the train and validation label mapping and records split metadata for those two splits only. Model fitting uses only `train.csv`; validation metrics use only `val.csv`. The test split is not loaded or inspected.

In [ ]:
train, val, split_metadata = load_data(paths.data_dir)
validate_data(train, val)
X_train, y_train, X_val, y_val, feature_columns, missing_summary, infinite_summary = prepare_features(train, val)

shape_table = pd.DataFrame(
    [
        {"Split": "train", "Rows": len(train), "Columns": train.shape[1]},
        {"Split": "val", "Rows": len(val), "Columns": val.shape[1]},
    ]
)
display(shape_table)
display(Markdown("**Feature columns**: " + ", ".join(feature_columns)))
display(train[["class", "class_encoded"]].drop_duplicates().sort_values("class_encoded"))

In [ ]:
display(split_metadata.assign(Percent=lambda df: (df["Percent"] * 100).round(2)))
display(missing_summary)
display(infinite_summary)

## Metrics Protocol

Balanced accuracy is the primary metric because the classes are moderately imbalanced and each class should contribute equally to model selection. Macro F1 is the main secondary metric and tie-breaker because it combines class-wise precision and recall. Accuracy, macro precision, macro recall, weighted F1, per-class metrics and confusion matrices are reported for every model with `zero_division=0` where applicable.

## Train and Evaluate Baselines

The benchmark trains exactly three fixed baselines: balanced multinomial Logistic Regression, balanced-subsample Random Forest and balanced multiclass CatBoost. No hyperparameter optimization is performed.

In [ ]:
result = run_benchmark(repo_root)
summary = result.summary_metrics[SUMMARY_COLUMNS]
per_class = result.per_class_metrics

macro_f1_winner = summary.sort_values(["Macro F1", "Balanced Accuracy"], ascending=False).iloc[0]
accuracy_winner = summary.sort_values(["Accuracy", "Balanced Accuracy"], ascending=False).iloc[0]
catboost_qso_recall = per_class[(per_class["Model"] == "CatBoost") & (per_class["Class"] == "QSO")]["Recall"].iloc[0]
rf_qso_recall = per_class[(per_class["Model"] == "Random Forest") & (per_class["Class"] == "QSO")]["Recall"].iloc[0]

display(summary.style.format({column: "{:.4f}" for column in SUMMARY_COLUMNS if column != "Model"}))
display(per_class.style.format({"Precision": "{:.4f}", "Recall": "{:.4f}", "F1": "{:.4f}"}))
display(Markdown(f"**Winner by balanced accuracy:** {result.best_model}"))
display(Markdown(f"**Winner by macro F1:** {macro_f1_winner['Model']}"))
display(Markdown(f"**Winner by accuracy:** {accuracy_winner['Model']}"))
display(Markdown(f"CatBoost QSO recall: `{catboost_qso_recall:.4f}`; Random Forest QSO recall: `{rf_qso_recall:.4f}`."))

## Saved Visualizations

The following figures are generated by the command-line benchmark and displayed from the saved PNG artifacts.

In [ ]:
for key, path in result.figure_paths.items():
    display(Markdown(f"### {path.name}"))
    display(Image(filename=str(path)))

## Benchmark Interpretation

CatBoost is selected as the current benchmark because it has the highest validation balanced accuracy, the predefined primary metric. Its advantage is partly driven by stronger recall for the minority `QSO` class. Random Forest has higher macro F1 and overall accuracy, so it has a better overall precision-recall balance and remains a strong alternative rather than being uniformly worse.

Future custom models must use the same prepared features and label mapping, fit only on `train.csv` during development, rank candidates on `val.csv`, inspect per-class metrics and confusion matrices, and leave the test split untouched until one final evaluation after model selection.

The feature-distribution plot uses visualization-only clipping at train-set 0.5th and 99.5th percentiles. Original feature values are preserved for model training.

In [ ]:
display(Markdown("The test split was not loaded or inspected. It remains reserved for the single final evaluation after model selection."))
display(Markdown(f"Report: `{result.report_path.relative_to(repo_root).as_posix()}`"))
display(Markdown("Tables: `reports/tables/baseline_metrics.csv`, `reports/tables/per_class_metrics.csv`"))

## References

- Dataset: https://www.kaggle.com/datasets/fedesoriano/stellar-classification-dataset-sdss17
- Logistic Regression and Random Forest precedent: https://github.com/wonhee3472/Stellar_Classification
- Random Forest precedent: https://www.kaggle.com/code/leylamammadzada/stellar-classification-random-forest
- CatBoost precedent: https://www.kaggle.com/code/aerdem/stellar-classification-catboost-97-accuracy